In [ ]:
using PhDProject

In [ ]:
function vars()
    P = Default_Spin_Boson_P_initialisation()
    P.N_L = 0                   
    P.N_R = 400                     
    P.Ns = 1                                                     
    P.N_chain = 40
    P.μ_L = 0                
    P.μ_R = 0              
    P.SB_coupling_op = "Sz"
    P.bosonic_cutoff = 4                  
    P.Δ = 1           
    P.symmetry_subspace = "Full"        
    P.compute_maps_bool = false
    P.using_system_ancilla = false
    P.S_ohmic = 1
    P.ohmic_cutoff = "exponential"
    P.method_type = "TFCM"
    P.n = 10                      
    P.δt = 0.1                   
    P.T = 1                  
    P.tdvp_cutoff = 1e-9   
    P.Kr_cutoff = 1e-9       
    P.T_enrich = 1   
    maxlinkdim = 2
    minlinkdim = 100
    P.β_cutoff = 1000 

    P.β_L= 1000
    P.β_R = 1000
    P.Γ_R = 0.02
    P.ωc = 7.5
    P.D = 50
    P.ωq = 0.5
    P.Δ = 1  
    return P
end
P = vars()
P.using_system_ancilla = true
P.compute_maps_bool = true
P.spec_fun_type = "Ohmic"
P.β_cutoff = 100
P.N_R = 250
P.β_R = 1000
P.Γ_R = 0.25#0.02
P.ωc = 1
P.ωq = 0.5
P.Δ = 1  
P.D = 10  
P.T = 40
P.T_enrich = 5
P.n = 5
P.minbonddim = 5
P.maxbonddim = 300
P.tdvp_cutoff = 1e-7  
P.Kr_cutoff = 1e-7  

P.S_ohmic = 0.1
DP = DP_initialisation(P);
P.T = 5
ψ,obs,L_vec,Λ_vec,ρ_vec,NESS_times = propagate_MPS_2(P,DP;use_spin_operators=true)


In [ ]:
NESS_times = range(P.n*P.δt,stop=10,step=P.n*P.δt)
spectra_L,NESS_list_L,σx_NESS_L,σy_NESS_L,σz_NESS_L = NESS_calculations(L_vec,DP.qS[1],"L",P,DP);
spectra_Λ,NESS_list_Λ,σx_NESS_Λ,σy_NESS_Λ,σz_NESS_Λ = NESS_calculations(Λ_vec,DP.qS[1],"Λ",P,DP);
dLdt,dρLdt,dρΛdt,dL_average_dt,dΛdt = convergence_analysis(L_vec,Λ_vec,P,DP;calculate_NESS_observables=false,take_symmetry_subset=false)

display(Plots.plot(1:length(dρLdt),dρLdt,label=L"
 
"))
display(Plots.plot(1:length(dLdt),dLdt,label=L"
 
"))

display(Plots.plot(real.(spectra_L),label=false,title="real spectrum of L"))
display(Plots.plot(real.(spectra_Λ),label=false,title="real spectrum of Λ"))


display(Plots.scatter(imag.(spectra_L),label=false,title="imaginary spectrum of L"))
display(Plots.scatter(imag.(spectra_Λ),label=false,title="imaginary spectrum of Λ"))


display(Plots.plot(abs.(spectra_L),label=false,title="abs spectrum of L"))
display(Plots.plot(abs.(spectra_Λ),label=false,title="abs spectrum of Λ"))


Plots.plot(NESS_times,real.(σx_NESS_L))
Plots.plot!(NESS_times,real.(σy_NESS_L))
display(Plots.plot!(NESS_times,real.(σz_NESS_L)))
Plots.plot(NESS_times,real.(σx_NESS_Λ))
Plots.plot!(NESS_times,real.(σy_NESS_Λ))
Plots.plot!(NESS_times,real.(σz_NESS_Λ))

In [ ]:
for i =reverse(1:length(L_vec))
    NESS = eigen(L_vec[i]).vectors[:,end]
    NESS = vectorise_mat(unvectorise_ρ(NESS,true))
    ρf = unvectorise_ρ(pinv(Λ_vec[i])*NESS,true)

    if sum(real.(eigen(ρf).values) .<0) == 0
        @show(i)
        @show(eigen(ρf).values)
    end
end

In [ ]:
function extrapolation_vs_map_propagation(ρ_init,L_vec,Λ_vec,memory_time,NESS_times)

    memory_time_ind = Int(memory_time/(P.n*P.δt))

    #ρ_init = vectorise_ρ(DP.ψ_init,P,DP;use_spin_operators=true)
    #ρ_init = [0,0,0,1]#[1 0;0 0]


    ρ_vec1 = Vector{Any}(undef,length(NESS_times)+1)
    σz_vec1 = zeros(length(ρ_vec1))
    σx_vec1= zeros(length(ρ_vec1))
    coherence_vec1 = complex(similar(σz_vec1))
    coherence_vec2 = complex(similar(σz_vec1))

    ρ_vec2 = Vector{Any}(undef,length(NESS_times)+1)
    σz_vec2 = zeros(length(ρ_vec1))
    σx_vec2= zeros(length(ρ_vec1))

    ρ_diff = similar(ρ_vec1)

    ρ_vec1[1] = ρ_init
    ρ_vec2[1] = ρ_init
    ρ_diff[1] = 0
    σz_vec1[1] = 0.5*real(ρ_init[1]-ρ_init[4])
    σx_vec1[1] = 0.5*real(ρ_init[2]+ρ_init[3])
    σz_vec2[1] = 0.5*real(ρ_init[1]-ρ_init[4])
    σx_vec2[1] = 0.5*real(ρ_init[2]+ρ_init[3])
    coherence_vec1[1] = ρ_init[2]
    coherence_vec2[1] = ρ_init[2]
    for i =1:length(NESS_times)
        
        ρ1= Λ_vec[i]*ρ_init
        if NESS_times[i] >=memory_time
            ρ2 = exp(L_vec[memory_time_ind]*P.n*P.δt)*ρ_vec2[i]
        else
            ρ2 = Λ_vec[i]*ρ_init
            
        end

        σz_vec1[i+1] = 0.5*real(ρ1[1]-ρ1[4])
        σx_vec1[i+1] = 0.5*real(ρ1[2]+ρ1[3])
        σz_vec2[i+1] = 0.5*real(ρ2[1]-ρ2[4])
        σx_vec2[i+1] = 0.5*real(ρ2[2]+ρ2[3])
        coherence_vec1[i+1] = ρ1[2]
        coherence_vec2[i+1] = ρ2[2]
        ρ_diff[i+1] = norm(ρ1-ρ2)
        ρ_vec1[i+1] = ρ1
        ρ_vec2[i+1] = ρ2
    end
    Plots.plot([0;NESS_times],σz_vec1,label="σz1")
    Plots.plot!([0;NESS_times],σz_vec2,label="σz2")
    # Plots.plot!([0;NESS_times],σx_vec1,label="σx1")
    # display(Plots.plot!([0;NESS_times],σx_vec2,label="σx2",legend = :right))
    # display(Plots.plot([0;NESS_times],ρ_diff))
    return σz_vec1,coherence_vec1,coherence_vec2,ρ_vec1
end
# Data_dict = load_object(dirname(dirname(@__DIR__))*"\multitime correlators using NMQMpE paper\Bosonic data\High T data 25_07_2025")
# L_vec = Data_dict["L_vec"]
# Λ_vec = Data_dict["Λ_vec"]
# NESS_times = Data_dict["NESS_times"]
# P,DP = Data_dict["Structs"]

NESS_times = range(P.n*P.δt,stop=40,step=P.n*P.δt)
memory_time = 10
memory_time_ind = 20#9#250#Int(memory_time/(P.n*P.δt))
NESS = eigen(L_vec[memory_time_ind]).vectors[:,end]
NESS = vectorise_mat(unvectorise_ρ(NESS,true))
ρf = vectorise_mat(unvectorise_ρ(pinv(Λ_vec[memory_time_ind])*NESS,true))
ρ_init = ρf#NESS#[1,0,0,0]
σz_vec1,coherence_vec1,coherence_vec1_ex,ρ_vec1 = extrapolation_vs_map_propagation(ρ_init,L_vec,Λ_vec,memory_time,NESS_times)
σz_vec2,coherence_vec2,coherence_vec2_ex,ρ_vec2 = extrapolation_vs_map_propagation(NESS,L_vec,Λ_vec,memory_time,NESS_times)
σz_vec3,coherence_vec3,coherence_vec3_ex,ρ_vec3 = extrapolation_vs_map_propagation([1,0,0,0],L_vec,Λ_vec,memory_time,NESS_times)
Plots.plot([0;NESS_times],σz_vec1,label= "fast state")
Plots.plot!([0;NESS_times],σz_vec2,label="NESS")
display(Plots.plot!([0;NESS_times],σz_vec3))

Plots.plot([0;NESS_times],real.(coherence_vec1),label= "fast state")
Plots.plot!([0;NESS_times],real.(coherence_vec2),label="NESS")
display(Plots.plot!([0;NESS_times],real.(coherence_vec3)))
Plots.plot!([0;NESS_times],real.(coherence_vec1_ex),label= "fast state")
Plots.plot!([0;NESS_times],real.(coherence_vec2_ex),label="NESS")
display(Plots.plot!([0;NESS_times],real.(coherence_vec3_ex),title="comparison"))


Plots.plot([0;NESS_times],imag.(coherence_vec1),label= "fast state")
Plots.plot!([0;NESS_times],imag.(coherence_vec2),label="NESS")
display(Plots.plot!([0;NESS_times],imag.(coherence_vec3)))


#display(Plots.plot!(NESS_times,real.(σz_NESS_L)/2,label="FP"))

#display(Plots.plot(real.(output_dict["partial_energies_vec"])))
     

In [ ]:
function extrapolation_vs_map_propagation_with_incremental_maps(ρ_init,L_vec,Λ_vec,dΛ_vec,memory_time,NESS_times)

    memory_time_ind = Int(memory_time/(P.n*P.δt))

    #ρ_init = vectorise_ρ(DP.ψ_init,P,DP;use_spin_operators=true)
    #ρ_init = [0,0,0,1]#[1 0;0 0]


    ρ_vec_Λ,ρ_vec_L,ρ_vec_dΛ= Vector{Any}(undef,length(NESS_times)+1),Vector{Any}(undef,length(NESS_times)+1),Vector{Any}(undef,length(NESS_times)+1)
    σz_vec_Λ,σz_vec_L,σz_vec_dΛ = zeros(length(ρ_vec_Λ)),zeros(length(ρ_vec_Λ)),zeros(length(ρ_vec_Λ))
    σx_vec_Λ,σx_vec_L,σx_vec_dΛ= zeros(length(ρ_vec_Λ)),zeros(length(ρ_vec_Λ)),zeros(length(ρ_vec_Λ))

    ρ_diff_L = similar(ρ_vec_Λ)
    ρ_diff_dΛ = similar(ρ_vec_Λ)

    ρ_vec_Λ[1] = ρ_init
    ρ_vec_L[1] = ρ_init
    ρ_vec_dΛ[1] = ρ_init

    ρ_diff_L[1] = 0
    ρ_diff_dΛ[1] = 0

    σz_vec_Λ[1] = 0.5*real(ρ_init[1]-ρ_init[4])
    σx_vec_Λ[1] = 0.5*real(ρ_init[2]+ρ_init[3])

    σz_vec_L[1] = 0.5*real(ρ_init[1]-ρ_init[4])
    σx_vec_L[1] = 0.5*real(ρ_init[2]+ρ_init[3])

    σz_vec_dΛ[1] = 0.5*real(ρ_init[1]-ρ_init[4])
    σx_vec_dΛ[1] = 0.5*real(ρ_init[2]+ρ_init[3])

    for i =1:length(NESS_times)
        
        ρΛ= Λ_vec[i]*ρ_init
        if NESS_times[i] >=memory_time
            ρL = exp(L_vec[memory_time_ind]*P.n*P.δt)*ρ_vec_L[i]
            ρdΛ = dΛ_vec[memory_time_ind]*ρ_vec_dΛ[i]
        else
            ρL= Λ_vec[i]*ρ_init
            ρdΛ= Λ_vec[i]*ρ_init
        end

        σz_vec_Λ[i+1] = 0.5*real(ρΛ[1]-ρΛ[4])
        σx_vec_Λ[i+1] = 0.5*real(ρΛ[2]+ρΛ[3])

        σz_vec_L[i+1] = 0.5*real(ρL[1]-ρL[4])
        σx_vec_L[i+1] = 0.5*real(ρL[2]+ρL[3])

        σz_vec_dΛ[i+1] = 0.5*real(ρdΛ[1]-ρdΛ[4])
        σx_vec_dΛ[i+1] = 0.5*real(ρdΛ[2]+ρdΛ[3])

        ρ_diff_L[i+1] = norm(ρΛ-ρL)
        ρ_diff_dΛ[i+1] = norm(ρΛ-ρdΛ)

        ρ_vec_Λ[i+1] = ρΛ
        ρ_vec_L[i+1] = ρL
        ρ_vec_dΛ[i+1] = ρdΛ
    end
    Plots.plot([0;NESS_times],σz_vec_Λ,label="σz_Λ",lw=2)
    Plots.plot!([0;NESS_times],σz_vec_L,label="σz_L",lw=2)
    Plots.plot!([0;NESS_times],σz_vec_dΛ,label="σz_dΛ",lw=2,linestyle=:dash)
    Plots.plot!([0;NESS_times],σx_vec_Λ,label="σx_Λ",lw=2)
    Plots.plot!([0;NESS_times],σx_vec_L,label="σx_L",lw=2)
    display(Plots.plot!([0;NESS_times],σx_vec_dΛ,label="σx_dΛ",legend = :right,lw=2,linestyle=:dash))
    Plots.plot([0;NESS_times],log10.(ρ_diff_L),label="error for L extrapolation",lw=2)
    display(Plots.plot!([0;NESS_times],log10.(ρ_diff_dΛ),label="error for dΛ extrapolation",lw=2))

end
function incremental_maps(Λ_vec)
    Λinv_vec = [get_inv(Λ) for Λ in Λ_vec]
    dΛ_vec = similar(Λ_vec)
    dΛ_vec[1] = Λ_vec[1]
    for i=2:length(Λ_vec)
        dΛ_vec[i] = Λ_vec[i]*Λinv_vec[i-1]
    end
    return dΛ_vec
end
dΛ_vec = incremental_maps(Λ_vec)
memory_time = 15
memory_time_ind = Int(memory_time/(P.n*P.δt))
NESS = eigen(L_vec[memory_time_ind]).vectors[:,end]
NESS = vectorise_mat(unvectorise_ρ(NESS,true))
ρf = vectorise_mat(unvectorise_ρ(pinv(Λ_vec[memory_time_ind])*NESS,true))
ρ_init = [1,0,0,0]#ρf#NESS#[1,0,0,0]
extrapolation_vs_map_propagation_with_incremental_maps(ρ_init,L_vec,Λ_vec,dΛ_vec,memory_time,NESS_times)
     


In [ ]:
function SB_lindblad_rates(L_vec)
    #Using formula A7 in https://journals.aps.org/pra/pdf/10.1103/PhysRevA.89.042120

    #For the spin boson, the G matrices are given by the pauli operators and the identity
    G0 = (1/√2)*[1 0;0 1]
    G1 = [1 0;0 -1]
    G2 = [0 1;1 0]
    G3 = [0 -im;im 0]
    G_matrices = [G0,G1,G2,G3]
    decoherence_matrix_vector = Vector{Any}(undef,length(L_vec))
    lindblad_rates_vector = complex(zeros(length(L_vec),3))
    @showprogress for (n,L) in enumerate(L_vec)
        decoherence_matrix = complex(zeros(3,3))
        for i = 1:3
            for j=1:3
                dij = 0
                for m=1:4
                    ##apply propagator to Gm
                    op_t = unvectorise_ρ(L*vectorise_mat(G_matrices[m]),false)
                    operator = G_matrices[m]*G_matrices[i+1]*op_t*G_matrices[j+1]
                    dij += tr(operator)
                end
                decoherence_matrix[i,j] = dij
            end
        end
        decoherence_matrix_vector[n] = decoherence_matrix
        lindblad_rates_vector[n,:] = eigen(decoherence_matrix).values
    end
    return lindblad_rates_vector
end
lindblad_rates_vector = SB_lindblad_rates(L_vec)
Plots.plot(real.(lindblad_rates_vector),ylim=(-1,1))

In [ ]:

function ρ_diff_average_SB(ρ_perturbed_vec,spin_corr_t,NESS_times,L_vec,operator_mat,P)
    (;δt,n,sys_mode_type) = P
    ρ_diff_average_vec = similar(NESS_times)
    corr_diff_average_vec = similar(NESS_times)
    corr_diff_max_vec = similar(NESS_times)
    @show(length(NESS_times))
    for (i,extrapolation_time) in enumerate(NESS_times)
        start_ind = Int(round(extrapolation_time/P.δt))
        start_L_ind = Int(round(extrapolation_time/(P.n*P.δt)))
        times2 = range(start_ind/10,stop=end_time,step=0.1)

        Lτm = L_vec[start_L_ind]
        

        ρ_perturbed = ρ_perturbed_vec[start_ind]
        ρ_perturbed_vec_test = Vector{Any}(undef,length(times))
        spin_corr_t_test = 0*similar(spin_corr_t)
        spin_corr_t_test[1:start_ind] = spin_corr_t[1:start_ind]
        ρ_perturbed_vec_test[1:start_ind] = ρ_perturbed_vec[1:start_ind]

        for i=1:length(times2)
            ρ_perturbed = exp(Lτm*P.δt)*ρ_perturbed
            spin_corr_t_test[start_ind+i] = tr(operator_mat*unvectorise_ρ(ρ_perturbed,false))
            ρ_perturbed_vec_test[start_ind+i] = ρ_perturbed
        end
        ρ_diff = [norm(ρ_perturbed_vec[i]-ρ_perturbed_vec_test[i]) for i=start_ind:length(ρ_perturbed_vec)]
        corr_diff = [norm(spin_corr_t[i]-spin_corr_t_test[i]) for i=start_ind:length(spin_corr_t)]
        ρ_diff_average_vec[i] = sum(ρ_diff)/(length(ρ_diff))
        corr_diff_average_vec[i] = sum(corr_diff)/(length(corr_diff))
        corr_diff_max_vec[i] = maximum(corr_diff)
    end
    return ρ_diff_average_vec,corr_diff_average_vec,corr_diff_max_vec
end
ρ_perturbed_vec = output_dict["ρ_perturbed_vec"][1:301]
L_vec = output_dict["L_vec"]
spin_corr_t = output_dict["spin_corr_t"]
Sz= 0.5*[1 0 ;0 -1]
Sx = 0.5*[0 1;1 0]
operator_mat = Sz
NESS_times = range(P.n*P.δt,stop=30,step=P.n*P.δt)
ρ_diff_average_vec,corr_diff_average_vec,corr_diff_max_vec = ρ_diff_average_SB(ρ_perturbed_vec,spin_corr_t,NESS_times,L_vec,operator_mat,P)

Plots.plot(legendfontsize=10,labelfontsize=15,ylabel= L"
",xlabel=L"
",lw=2)
display(Plots.plot!(NESS_times,log10.(ρ_diff_average_vec),ylim=(-5,0)))
display(Plots.plot(NESS_times,(ρ_diff_average_vec)))

     
length(NESS_times) = 60
